In [ ]:
from pathlib import Path
import re

from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

import pandas as pd

import src

In [ ]:
pd.set_option("display.max_colwidth", 512)

In [ ]:
nltk.download("vader_lexicon", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("stopwords", quiet=True)

True

In [ ]:
analyzer = SentimentIntensityAnalyzer()

STOPWORDS = set(stopwords.words("english"))


def tokenize(text: str) -> list[str]:
    return word_tokenize(text, language="english")


def clean(text: str):

    # unify text
    text = text.strip().lower().replace("\n", " ")

    # remove urls
    text = re.sub(r"(?:\@|http?\://|https?\://|www)\S+", "", text)

    # remove non-words
    text = re.sub(r"[^\w\s]+|\d+|#\S+", "", text)

    # remove stopwords
    tokens = tokenize(text)
    tokens = [t for t in tokens if t not in STOPWORDS]

    return text


def analyze(analyzer, text: str):
    clean_text = clean(text)
    scores = analyzer.polarity_scores(clean_text)
    return scores["compound"]

In [ ]:
node_folder = src.PATH / "data/interim/node_lists/"
node_files = list(node_folder.iterdir())

In [ ]:
in_file = next(iter(node_files))

In [ ]:
out_folder = src.PATH / "data/interim/title_sentiments/"
out_folder.mkdir(exist_ok=True, parents=True)

In [ ]:
for file in node_files:

    filename = file.name
    out_file = out_folder / filename

    df = pd.read_csv(file)
    df["sentiment"] = df["title"].apply(lambda x: analyze(analyzer, x))

    df[["video_id", "sentiment"]].to_csv(out_file, index=False)